# 26 — Post-Release Validation

Validates the **published** model exactly the way users consume it: download
`ARO-Lang/aro-coder-4bit` from HuggingFace and drive it through the same path
`aro ask` uses. NB22's gates run at packaging time on **local** artifacts —
nothing before this stage ever exercised the uploaded weights, tokenizer and
bundled system prompt end-to-end, so regressions in that gap were discovered
by users.

Unlike NB24/NB25 (manual, stdin-driven chat notebooks), this stage is fully
**automated** and part of the meta pipeline: it runs a fixed set of scripted
probes (code generation + knowledge Q&A), checks the same promotion-gate
metrics as NB22, and records the outcome.

## Pipeline

```
1. Download the released model from HuggingFace (as `aro ask` would)
2. Load it with MLX + the bundled aro_system_prompt.txt
3. Run ~26 scripted probes through the `aro ask` generation path
4. Check promotion-gate metrics (reply rate, empty-think, syntax pass,
   tool leakage, URL contamination)
5. Record the result to release/eval/last-run.json and upload it
   alongside the model (eval/last-run.json on the HF repo)
```

**Input:**  `config.PREFERRED_MODEL_ID` (the published repo)
**Output:** `release/eval/last-run.json` (+ `eval/last-run.json` on the Hub)
**Fails:** raises when the published model breaches any gate — the meta
pipeline marks the release run failed.

## Setup

In [ ]:
import sys, importlib, json, re, subprocess, tempfile, time
from pathlib import Path
from collections import Counter

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

RELEASE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUT_DIR  = RELEASE_DIR / 'eval'
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
LAST_RUN_PATH = EVAL_OUT_DIR / 'last-run.json'

# Generation settings — mirror the `aro ask` CLI defaults (see NB25).
MAX_TOKENS  = 1024
TEMPERATURE = 0.3
TOP_P       = 0.9

# Promotion-gate thresholds — identical to NB22 so "passed at packaging"
# and "works when published" are measured with the same yardstick.
GATE_REPLY_RATE_MIN  = 0.50   # at least 50% must produce non-empty content
GATE_EMPTY_THINK_MAX = 0.20   # at most 20% may collapse to empty <think></think>
GATE_SYNTAX_PASS_MIN = 0.40   # at least 40% of complete code programs must pass aro check
GATE_TOOL_LEAK_MAX   = 0.02   # at most 2% of replies may contain tool-call leakage
GATE_URL_CONTAM_MAX  = 0.05   # at most 5% of replies may contain markdown image URLs

print(f'Published model: {PREFERRED_MODEL_ID}')
print(f'Result file:     {LAST_RUN_PATH}')

## Step 1 — Download the published model (as a user would)

`snapshot_download` is the same mechanism `aro ask` uses on first run. The
downloaded revision's commit hash is recorded so the validation result is
attributable to an exact Hub state.

In [ ]:
try:
    from huggingface_hub import snapshot_download, HfApi
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import snapshot_download, HfApi

if not _hf_model_exists(PREFERRED_MODEL_ID):
    raise RuntimeError(
        f'{PREFERRED_MODEL_ID} is not available on HuggingFace (no config.json). '
        f'Run NB22 with HF credentials to publish a release first.'
    )

print(f'Downloading {PREFERRED_MODEL_ID} from HuggingFace...')
_t0 = time.time()
SNAPSHOT_PATH = Path(snapshot_download(PREFERRED_MODEL_ID))
print(f'Snapshot ready in {time.time() - _t0:.0f}s: {SNAPSHOT_PATH}')

# Record the exact Hub revision being validated.
_hub_commit = None
try:
    _hub_commit = HfApi().model_info(PREFERRED_MODEL_ID).sha
    print(f'Hub revision: {_hub_commit}')
except Exception as _e:
    print(f'Could not resolve Hub revision ({_e!s:.80}) — recording snapshot path only.')

# Release version being validated (written by NB22 into the manifest).
_manifest_path = RELEASE_DIR / 'model_manifest.json'
RELEASE_VERSION = None
if _manifest_path.exists():
    try:
        RELEASE_VERSION = json.loads(_manifest_path.read_text()).get('version')
        print(f'Manifest version: v{RELEASE_VERSION}')
    except Exception:
        pass

## Step 2 — Load via the `aro ask` path

Uses the **bundled** `aro_system_prompt.txt` from the downloaded snapshot when
present — the same file `ContextStore.swift` resolves from the HF cache — so
the probes see exactly the prompt users get. Falls back to
`build_system_prompt()` only if the bundle is missing (and warns, because
that itself is a release defect).

In [ ]:
load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

print(f'Loading {PREFERRED_MODEL_ID} (from HF cache)...')
model, tokenizer = load_fn(PREFERRED_MODEL_ID)
print('Model loaded.')

_bundled_prompt = SNAPSHOT_PATH / 'aro_system_prompt.txt'
if _bundled_prompt.exists():
    SYSTEM_PROMPT = _bundled_prompt.read_text()
    _prompt_source = 'bundled aro_system_prompt.txt'
else:
    print('WARNING: released model has no bundled aro_system_prompt.txt — '
          '`aro ask` cannot match the training-time prompt. Falling back to '
          'build_system_prompt(); fix the NB22 bundling step.')
    kb = load_knowledge()
    SYSTEM_PROMPT = build_system_prompt(kb)
    _prompt_source = 'build_system_prompt() fallback'

print(f'System prompt: {len(SYSTEM_PROMPT)} chars ({_prompt_source})')


def ask(prompt, max_tokens=MAX_TOKENS, temperature=TEMPERATURE):
    """One `aro ask` style turn: system prompt + user prompt → reply."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    return generate_fn(
        model, tokenizer,
        prompt=tokens,
        max_tokens=max_tokens,
        sampler=make_sampler_fn(temp=temperature, top_p=TOP_P),
        verbose=False,
    ).strip()

## Step 3 — Scripted probes

~26 fixed probes covering code generation (REPL one-liners, HTTP APIs, event
handlers, repositories, full applications, user-defined actions, debugging)
and knowledge Q&A. Code replies containing a complete program are validated
with `aro check`; every reply is scanned for empty-think collapse, tool-name
leakage and URL contamination — the same checks as NB22's promotion gate.

In [ ]:
PROBES = [
    # ── Code generation ────────────────────────────────────────────────────
    ('code', 'repl_oneliner',  "Write a single ARO line that logs 'Hello, World!' to the console."),
    ('code', 'repl_block',     'Write ARO statements that create a list of three numbers and log its length.'),
    ('code', 'application',    'Write a minimal ARO application that logs Hello World and exits.'),
    ('code', 'application',    'Write an ARO Application-Start that starts an HTTP server and keeps running with Keepalive.'),
    ('code', 'application',    'Write an ARO Application-End: Success handler that logs a shutdown message.'),
    ('code', 'http_api',       'Write an ARO feature set named getUser that retrieves a user by ID from the user-repository and returns it as an OK response.'),
    ('code', 'http_api',       'Write an ARO feature set named createOrder that extracts the request body, stores a new order, and returns a Created status.'),
    ('code', 'http_api',       'Write an ARO feature set named listProducts that retrieves all products and returns them.'),
    ('code', 'event_handler',  'Write an ARO event handler that sends a welcome email when a UserCreated event is received.'),
    ('code', 'event_handler',  'Write an ARO feature set that emits an OrderPlaced event with the order data.'),
    ('code', 'repository',     'Write an ARO feature set that observes the user-repository and logs every change.'),
    ('code', 'repository',     'Write an ARO feature set that stores a new record in the sessions-repository and emits a SessionCreated event.'),
    ('code', 'iteration',      'Write an ARO for-each loop that logs each item in a list of names.'),
    ('code', 'conditional',    'Write an ARO feature set with a When guard that only returns OK when the user role is "admin".'),
    ('code', 'computation',    'Write an ARO feature set that computes the total price from price and quantity and returns it.'),
    ('code', 'user_action',    'Define an ARO user-defined action named DoubleValue that doubles a number.'),
    ('code', 'debugging',      'Fix this ARO code so it passes aro check:\n\n```aro\n(listUsers: User API) {\n    Retrieve the <users> from the <user-repository>.\n    Return an <OK: status> with <users>.\n```'),
    ('code', 'file',           'Write an ARO Application-Start that reads a config file and logs its contents.'),
    # ── Knowledge Q&A ──────────────────────────────────────────────────────
    ('qa', 'explain', 'What is the difference between Extract and Compute actions in ARO?'),
    ('qa', 'explain', 'How do I emit a custom event in ARO and handle it in another feature set?'),
    ('qa', 'explain', 'How does the Publish action work in ARO? Give an example.'),
    ('qa', 'explain', 'Why are ARO variables immutable and how do I update a value?'),
    ('qa', 'explain', 'What does the Keepalive action do and when do I need it?'),
    ('qa', 'explain', 'How does contract-first HTTP routing work in ARO?'),
    ('qa', 'edge',    'Does ARO have error handling? Explain the error philosophy briefly.'),
    ('qa', 'edge',    'What is a .store file in ARO and when is it writable?'),
]

_THINK_RE = re.compile(r'<think>[\s\S]*?</think>', re.S)
_FENCE_RE = re.compile(r'```aro\s*\n(.*?)```', re.S)
_TOOL_RE  = re.compile(r'\b(read_file|write_file|edit_file|aro_check|aro_run|aro_test|create_plugin|list_files)\s*\(')
_URL_RE   = re.compile(r'!\[[^\]]*\]\(\s*https?://[^\s)]+\s*\)')


def aro_check(code):
    """Run `aro check`; returns (passed, first_error_line)."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            if r.returncode == 0:
                return True, ''
            err = (r.stderr or r.stdout).strip()
            return False, (err.splitlines()[0][:160] if err else 'aro check failed')
    except Exception as e:
        return False, f'exception: {e}'[:160]


results = []
_replies = 0
_empty_think = 0
_with_code = 0
_syntax_pass = 0
_tool_leak = 0
_url_contam = 0
_qa_answered = 0
_qa_total = 0

print(f'Running {len(PROBES)} probes against the published model...\n')
_t0 = time.time()

for _idx, (_mode, _cat, _prompt) in enumerate(PROBES, 1):
    _raw = ask(_prompt)
    _stripped = _THINK_RE.sub('', _raw).strip()
    _is_empty_think = ('<think>' in _raw and not _stripped)
    if _is_empty_think:
        _empty_think += 1
    if _stripped:
        _replies += 1

    _blocks = [b.strip() for b in _FENCE_RE.findall(_stripped) if b.strip()]
    _has_code = bool(_blocks)
    _check_pass = None
    _check_err = ''
    if _has_code and is_complete_program(_blocks):
        _with_code += 1
        _check_pass, _check_err = aro_check('\n\n'.join(_blocks))
        if _check_pass:
            _syntax_pass += 1

    if _mode == 'qa':
        _qa_total += 1
        _text_only = re.sub(r'```.*?```', '', _stripped, flags=re.DOTALL).strip()
        if len(_text_only) > 30:
            _qa_answered += 1

    _leak = bool(_TOOL_RE.search(_stripped))
    _contam = bool(_URL_RE.search(_stripped))
    if _leak:
        _tool_leak += 1
    if _contam:
        _url_contam += 1

    if _check_pass is True:
        _status = 'PASS'
    elif _check_pass is False:
        _status = 'FAIL'
    elif _mode == 'code':
        _status = 'NO PROGRAM'
    else:
        _status = 'answered' if len(re.sub(r'```.*?```', '', _stripped, flags=re.DOTALL).strip()) > 30 else 'thin'

    results.append({
        'idx': _idx, 'mode': _mode, 'cat': _cat, 'prompt': _prompt,
        'reply': _stripped[:500],
        'empty_think': _is_empty_think,
        'has_code': _has_code,
        'aro_check_pass': _check_pass,
        'check_error': _check_err,
        'tool_leak': _leak,
        'url_contam': _contam,
    })
    print(f'  [{_idx:2d}/{len(PROBES)}] [{_mode:4s}/{_cat:<13}] {_status:<11} {_prompt[:55]}')

print(f'\nDone in {(time.time() - _t0) / 60:.1f} min')

## Step 4 — Gate & record the result

Computes the promotion-gate metrics on the published model, writes
`release/eval/last-run.json`, uploads it alongside the model on the Hub
(`eval/last-run.json`), and **then** raises if any gate is breached — so a
failing run still leaves a full record.

In [ ]:
from datetime import datetime, timezone

_N = len(PROBES)
metrics = {
    'reply_rate':       _replies / _N,
    'empty_think_rate': _empty_think / _N,
    'syntax_pass_rate': _syntax_pass / _with_code if _with_code else 0.0,
    'tool_leak_rate':   _tool_leak / _N,
    'url_contam_rate':  _url_contam / _N,
    'qa_answer_rate':   _qa_answered / _qa_total if _qa_total else 0.0,
    'total': _N,
    'with_code': _with_code,
    'syntax_pass': _syntax_pass,
}

print('POST-RELEASE VALIDATION RESULTS:')
print(f"  reply rate:        {metrics['reply_rate']:.1%}  (gate ≥ {GATE_REPLY_RATE_MIN:.0%})")
print(f"  empty-think rate:  {metrics['empty_think_rate']:.1%}  (gate ≤ {GATE_EMPTY_THINK_MAX:.0%})")
print(f"  complete programs: {_with_code}/{_N}")
print(f"  syntax pass rate:  {metrics['syntax_pass_rate']:.1%}  (gate ≥ {GATE_SYNTAX_PASS_MIN:.0%})")
print(f"  tool-name leak:    {metrics['tool_leak_rate']:.1%}  (gate ≤ {GATE_TOOL_LEAK_MAX:.0%})")
print(f"  URL contamination: {metrics['url_contam_rate']:.1%}  (gate ≤ {GATE_URL_CONTAM_MAX:.0%})")
print(f"  Q&A answered:      {metrics['qa_answer_rate']:.1%}")

_breaches = []
if metrics['reply_rate'] < GATE_REPLY_RATE_MIN:
    _breaches.append(f"reply rate {metrics['reply_rate']:.1%} < {GATE_REPLY_RATE_MIN:.0%}")
if metrics['empty_think_rate'] > GATE_EMPTY_THINK_MAX:
    _breaches.append(f"empty-think collapse {metrics['empty_think_rate']:.1%} > {GATE_EMPTY_THINK_MAX:.0%}")
if _with_code > 0 and metrics['syntax_pass_rate'] < GATE_SYNTAX_PASS_MIN:
    _breaches.append(f"syntax pass {metrics['syntax_pass_rate']:.1%} < {GATE_SYNTAX_PASS_MIN:.0%}")
if metrics['tool_leak_rate'] > GATE_TOOL_LEAK_MAX:
    _breaches.append(f"tool-name leak {metrics['tool_leak_rate']:.1%} > {GATE_TOOL_LEAK_MAX:.0%}")
if metrics['url_contam_rate'] > GATE_URL_CONTAM_MAX:
    _breaches.append(f"URL contamination {metrics['url_contam_rate']:.1%} > {GATE_URL_CONTAM_MAX:.0%}")

last_run = {
    'model_id': PREFERRED_MODEL_ID,
    'hub_commit': _hub_commit,
    'version': RELEASE_VERSION,
    'validated_at': datetime.now(timezone.utc).isoformat(),
    'prompt_source': _prompt_source,
    'passed': not _breaches,
    'breaches': _breaches,
    'gates': {
        'reply_rate_min': GATE_REPLY_RATE_MIN,
        'empty_think_max': GATE_EMPTY_THINK_MAX,
        'syntax_pass_min': GATE_SYNTAX_PASS_MIN,
        'tool_leak_max': GATE_TOOL_LEAK_MAX,
        'url_contam_max': GATE_URL_CONTAM_MAX,
    },
    'metrics': metrics,
    'results': results,
}
LAST_RUN_PATH.write_text(json.dumps(last_run, indent=2))
print(f'\nResult recorded: {LAST_RUN_PATH}')

# ── Upload the record alongside the model (eval/last-run.json on the Hub) ──
# Same auth resolution as NB22: never hang waiting for credentials.
import os
from huggingface_hub import whoami
try:
    from huggingface_hub import get_token as _hf_get_token
except ImportError:
    from huggingface_hub import HfFolder
    _hf_get_token = HfFolder.get_token
_hf_token = os.environ.get('HF_TOKEN') or _hf_get_token()
if _hf_token:
    try:
        whoami(token=_hf_token)
        HfApi().upload_file(
            path_or_fileobj=str(LAST_RUN_PATH),
            path_in_repo='eval/last-run.json',
            repo_id=PREFERRED_MODEL_ID,
            repo_type='model',
            commit_message=f'Post-release validation: '
                           f'{"PASSED" if not _breaches else "FAILED"} '
                           f'({metrics["syntax_pass_rate"]:.0%} syntax pass)',
        )
        print(f'Uploaded: https://huggingface.co/{PREFERRED_MODEL_ID}/blob/main/eval/last-run.json')
    except Exception as _e:
        print(f'Could not upload eval record ({_e!s:.120}) — local record kept.')
else:
    print('No HF token — eval record kept locally only.')

del model
import gc; gc.collect()

if _breaches:
    raise RuntimeError(
        'POST-RELEASE VALIDATION FAILED — the published model breaches the '
        'promotion gates as users would experience it:\n  - ' +
        '\n  - '.join(_breaches) +
        f'\n\nSee {LAST_RUN_PATH} for per-probe details. Roll back to the '
        f'previous Hub tag (revision="v<previous>") or fix and re-release.'
    )

print('\nPOST-RELEASE VALIDATION PASSED — the published model works as released.')

In [ ]:
# ── Chart: probe outcomes + gate metrics ─────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '26_post_release_validation.png'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: code probe outcomes ────────────────────────────────────────
_code_pass = sum(1 for r in results if r['aro_check_pass'] is True)
_code_fail = sum(1 for r in results if r['aro_check_pass'] is False)
_code_nop  = sum(1 for r in results if r['mode'] == 'code' and r['aro_check_pass'] is None)
_vals   = [_code_pass, _code_fail, _code_nop]
_labels = [f'Pass\n({_code_pass})', f'Fail\n({_code_fail})', f'No program\n({_code_nop})']
_colors = ['#2ecc71', '#e74c3c', '#95a5a6']
_nz = [(v, l, c) for v, l, c in zip(_vals, _labels, _colors) if v > 0]
if _nz:
    v2, l2, c2 = zip(*_nz)
    ax1.pie(v2, labels=l2, colors=c2, autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax1.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    ax1.text(0, 0, f'{metrics["syntax_pass_rate"]:.0%}', ha='center', va='center',
             fontsize=16, fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'no code probes', ha='center', va='center', transform=ax1.transAxes)
ax1.set_title('Published Model — Code Probes (aro check)', fontsize=12, fontweight='bold')

# ── Right: gate metrics vs thresholds ────────────────────────────────
_gate_rows = [
    ('reply rate',   metrics['reply_rate'],       GATE_REPLY_RATE_MIN,  True),
    ('empty think',  metrics['empty_think_rate'], GATE_EMPTY_THINK_MAX, False),
    ('syntax pass',  metrics['syntax_pass_rate'], GATE_SYNTAX_PASS_MIN, True),
    ('tool leak',    metrics['tool_leak_rate'],   GATE_TOOL_LEAK_MAX,   False),
    ('URL contam',   metrics['url_contam_rate'],  GATE_URL_CONTAM_MAX,  False),
]
_g_labels = [g[0] for g in _gate_rows]
_g_vals   = [g[1] * 100 for g in _gate_rows]
_g_ok     = [(v >= t if hib else v <= t) for _, v, t, hib in _gate_rows]
_g_colors = ['#2ecc71' if ok else '#e74c3c' for ok in _g_ok]
y = range(len(_g_labels))
ax2.barh(list(y), _g_vals, color=_g_colors, edgecolor='white', height=0.55)
for _i, (_, _v, _t, _hib) in enumerate(_gate_rows):
    ax2.plot([_t * 100, _t * 100], [_i - 0.35, _i + 0.35], color='#333', lw=1.5, ls='--')
    ax2.text(_v * 100 + 2, _i, f'{_v:.0%}', va='center', fontsize=9)
ax2.set_yticks(list(y))
ax2.set_yticklabels(_g_labels, fontsize=10)
ax2.set_xlabel('Rate (%)  (dashed line = gate threshold)')
ax2.set_xlim(0, 110)
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)
ax2.set_title('Gate Metrics vs Thresholds', fontsize=12, fontweight='bold')

fig.suptitle(
    f'Post-Release Validation — {PREFERRED_MODEL_ID}'
    + (f'  ·  v{RELEASE_VERSION}' if RELEASE_VERSION else '')
    + f'  ·  {"PASSED" if not _breaches else "FAILED"}',
    fontsize=13, fontweight='bold', y=1.02,
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')